# Dice medium-color 데이터셋 전용 YOLO11s 학습

첫 번째 데이터셋만 80/10/10으로 분할하여 YOLO11s를 100에포크 학습합니다.

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CUDA 사용 불가")

In [ ]:
import random
from pathlib import Path

import yaml
from ultralytics import YOLO

# test.ipynb를 yacht_dice 프로젝트 폴더에서 열었다고 가정한다.
# 루트 또는 tests/notebooks에서 실행해도 프로젝트 폴더를 찾는다.
PROJECT_DIR = next(
    folder for folder in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (folder / "Dice.v2-medium-color.yolov11").is_dir()
)
DATASET_DIR = PROJECT_DIR / "Dice.v2-medium-color.yolov11"
IMAGES_DIR = DATASET_DIR / "export" / "images"
LABELS_DIR = DATASET_DIR / "export" / "labels"
SPLITS_DIR = PROJECT_DIR / "dataset_splits" / "dice_medium_only"
DATA_YAML = SPLITS_DIR / "dice_medium_only.yaml"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SEED = 42

if not IMAGES_DIR.is_dir() or not LABELS_DIR.is_dir():
    raise FileNotFoundError(f"데이터셋 폴더를 찾을 수 없습니다: {DATASET_DIR}")

# 이름이 같은 YOLO 라벨 파일이 존재하는 이미지만 사용한다.
images = sorted(
    path for path in IMAGES_DIR.iterdir()
    if path.suffix.lower() in IMAGE_EXTENSIONS
    and (LABELS_DIR / f"{path.stem}.txt").is_file()
)
if not images:
    raise RuntimeError("학습 가능한 이미지와 라벨 쌍이 없습니다.")

# 항상 같은 데이터가 각 분할에 들어가도록 시드를 고정한다.
random.Random(SEED).shuffle(images)
train_end = int(len(images) * 0.8)
val_end = train_end + int(len(images) * 0.1)
splits = {
    "train": images[:train_end],
    "val": images[train_end:val_end],
    "test": images[val_end:],
}

SPLITS_DIR.mkdir(parents=True, exist_ok=True)
for split_name, split_images in splits.items():
    (SPLITS_DIR / f"{split_name}.txt").write_text(
        "\n".join(str(path.resolve()) for path in split_images) + "\n",
        encoding="utf-8",
    )

source_config = yaml.safe_load((DATASET_DIR / "data.yaml").read_text(encoding="utf-8"))
data_config = {
    "train": str((SPLITS_DIR / "train.txt").resolve()),
    "val": str((SPLITS_DIR / "val.txt").resolve()),
    "test": str((SPLITS_DIR / "test.txt").resolve()),
    "nc": source_config["nc"],
    "names": source_config["names"],
}
DATA_YAML.write_text(
    yaml.safe_dump(data_config, allow_unicode=True, sort_keys=False),
    encoding="utf-8",
)

print(
    f"전체={len(images)}, 학습={len(splits['train'])}, "
    f"검증={len(splits['val'])}, 테스트={len(splits['test'])}"
)

# patience=0으로 설정하여 Early Stopping 없이 100에포크를 모두 수행한다.
model = YOLO("yolo11s.pt")
results = model.train(
    data=str(DATA_YAML),
    epochs=100,
    imgsz=640,
    batch=-1,
    device=0,
    workers=0,
    project=str(PROJECT_DIR / "runs"),
    name="dice_medium_yolo11s_e100",
    exist_ok=False,
    seed=SEED,
    patience=0,
    plots=True,
)

BEST_WEIGHTS = Path(results.save_dir) / "weights" / "best.pt"
print("Best 가중치:", BEST_WEIGHTS)
print("검증 혼돈행렬:", Path(results.save_dir) / "confusion_matrix.png")
print("정규화 검증 혼돈행렬:", Path(results.save_dir) / "confusion_matrix_normalized.png")

In [ ]:
# 위 학습 셀이 끝난 후 실행하면, 학습에 사용하지 않은 테스트셋으로 최종 평가한다.
best_model = YOLO(str(BEST_WEIGHTS))
test_metrics = best_model.val(
    data=str(DATA_YAML),
    split="test",
    imgsz=640,
    device=0,
    workers=0,
    project=str(PROJECT_DIR / "runs"),
    name="dice_medium_yolo11s_e100_test",
    plots=True,
)

TEST_DIR = Path(test_metrics.save_dir)
print("테스트 혼돈행렬:", TEST_DIR / "confusion_matrix.png")
print("정규화 테스트 혼돈행렬:", TEST_DIR / "confusion_matrix_normalized.png")
print("테스트 mAP50-95:", test_metrics.box.map)